# Project 2 Milestone 6: Candidate-Level Cross-Method Summary

This notebook builds a candidate-level evidence profile for the 27 Project 1 chemo-kinematic candidates. It combines Project 1 physical/candidate flags with PCA and UMAP embedding coordinates, baseline DBSCAN labels, and small-grid DBSCAN robustness evidence.

## Scientific question

Which Project 1 candidates show consistent evidence of being outliers or structurally distinct across PCA, UMAP, DBSCAN baseline clustering, and DBSCAN robustness tests?

Important caveat: these are feature-space outlier signals, not confirmations of independent astrophysical structures.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('../data/processed')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_PATH = DATA_DIR / 'gaia_lamost_larger_chemo_kinematic_candidates.csv'
PCA_PATH = DATA_DIR / 'project2_combined_chemo_kinematic_pca_embedding.csv'
UMAP_PATH = DATA_DIR / 'project2_combined_chemo_kinematic_umap_embedding.csv'
PCA_LABEL_PATH = DATA_DIR / 'project2_pca_dbscan_labels_baseline.csv'
UMAP_LABEL_PATH = DATA_DIR / 'project2_umap_dbscan_labels_baseline.csv'

OUT_CSV = DATA_DIR / 'project2_candidate_cross_method_summary.csv'
OUT_FIG = FIG_DIR / 'project2_candidate_cross_method_evidence_summary.png'

## Load upstream Project 1 and Project 2 products

In [ ]:
candidates = pd.read_csv(CANDIDATE_PATH)
pca = pd.read_csv(PCA_PATH)
umap = pd.read_csv(UMAP_PATH)
pca_labels = pd.read_csv(PCA_LABEL_PATH)
umap_labels = pd.read_csv(UMAP_LABEL_PATH)

print('candidates:', candidates.shape)
print('pca:', pca.shape)
print('umap:', umap.shape)
print('pca labels:', pca_labels.shape)
print('umap labels:', umap_labels.shape)

assert candidates['source_id'].is_unique
assert pca['source_id'].is_unique
assert umap['source_id'].is_unique
assert pca_labels['source_id'].is_unique
assert umap_labels['source_id'].is_unique

## Recompute M5 small-grid DBSCAN labels at object level

In [ ]:
SMALL_EPS_VALUES = [0.20, 0.25, 0.30]
SMALL_MIN_SAMPLES_VALUES = [5, 8, 12]

def run_dbscan(dataframe, coordinate_cols, eps, min_samples):
    X = dataframe[coordinate_cols].to_numpy()
    X_scaled = StandardScaler().fit_transform(X)
    model = DBSCAN(eps=eps, min_samples=min_samples)
    return model.fit_predict(X_scaled)

def object_level_noise_counts(dataframe, coordinate_cols, prefix):
    result = dataframe[['source_id']].copy()
    noise_cols = []
    for eps in SMALL_EPS_VALUES:
        for min_samples in SMALL_MIN_SAMPLES_VALUES:
            labels = run_dbscan(dataframe, coordinate_cols, eps=eps, min_samples=min_samples)
            col = f'{prefix}_noise_eps_{eps:.2f}_min_{min_samples}'
            result[col] = labels == -1
            noise_cols.append(col)
    result[f'{prefix}_noise_grid_count'] = result[noise_cols].sum(axis=1).astype(int)
    result[f'{prefix}_noise_grid_total'] = len(noise_cols)
    result[f'{prefix}_noise_stability_fraction'] = result[f'{prefix}_noise_grid_count'] / len(noise_cols)
    return result

pca_noise_counts = object_level_noise_counts(
    pca,
    ['combined_chemo_kinematic_pc1', 'combined_chemo_kinematic_pc2'],
    'pca',
)
umap_noise_counts = object_level_noise_counts(
    umap,
    ['umap_1', 'umap_2'],
    'umap',
)

pca_noise_counts.head()

## Build candidate-level cross-method summary table

In [ ]:
physical_cols = [
    'source_id',
    'feh',
    'tangential_velocity_kms',
    'galcen_vtot_kms',
    'bp_rp',
    'absolute_g_mag',
    'metal_poor_candidate',
    'high_vtan_candidate',
    'chemo_kinematic_candidate',
]

summary = candidates[physical_cols].copy()

summary = summary.merge(
    pca[['source_id', 'combined_chemo_kinematic_pc1', 'combined_chemo_kinematic_pc2']],
    on='source_id',
    how='left',
)
summary = summary.merge(
    umap[['source_id', 'umap_1', 'umap_2']],
    on='source_id',
    how='left',
)
summary = summary.merge(
    pca_labels[['source_id', 'pca_dbscan_label']],
    on='source_id',
    how='left',
)
summary = summary.merge(
    umap_labels[['source_id', 'umap_dbscan_label']],
    on='source_id',
    how='left',
)
summary = summary.merge(
    pca_noise_counts[['source_id', 'pca_noise_grid_count', 'pca_noise_grid_total', 'pca_noise_stability_fraction']],
    on='source_id',
    how='left',
)
summary = summary.merge(
    umap_noise_counts[['source_id', 'umap_noise_grid_count', 'umap_noise_grid_total', 'umap_noise_stability_fraction']],
    on='source_id',
    how='left',
)

summary['pca_dbscan_noise_baseline'] = summary['pca_dbscan_label'] == -1
summary['umap_dbscan_noise_baseline'] = summary['umap_dbscan_label'] == -1

summary['cross_method_noise_score'] = (
    summary['pca_dbscan_noise_baseline'].astype(int)
    + summary['umap_dbscan_noise_baseline'].astype(int)
    + summary['pca_noise_stability_fraction']
    + summary['umap_noise_stability_fraction']
)

summary = summary.sort_values(
    ['cross_method_noise_score', 'pca_noise_grid_count', 'umap_noise_grid_count', 'galcen_vtot_kms'],
    ascending=[False, False, False, False],
).reset_index(drop=True)

summary.to_csv(OUT_CSV, index=False)
print('Saved:', OUT_CSV)
print('shape:', summary.shape)
summary.head(10)

## Summary statistics

In [ ]:
print('Number of candidate rows:', len(summary))
print('PCA baseline noise candidates:', int(summary['pca_dbscan_noise_baseline'].sum()))
print('UMAP baseline noise candidates:', int(summary['umap_dbscan_noise_baseline'].sum()))
print('PCA small-grid noise count distribution:')
print(summary['pca_noise_grid_count'].value_counts().sort_index())
print('UMAP small-grid noise count distribution:')
print(summary['umap_noise_grid_count'].value_counts().sort_index())

summary[[
    'source_id', 'feh', 'tangential_velocity_kms', 'galcen_vtot_kms',
    'pca_dbscan_noise_baseline', 'umap_dbscan_noise_baseline',
    'pca_noise_grid_count', 'umap_noise_grid_count',
    'pca_noise_stability_fraction', 'umap_noise_stability_fraction',
    'cross_method_noise_score',
]].head(12)

## Candidate evidence figure

In [ ]:
plot_df = summary.copy()
plot_df['short_source_id'] = plot_df['source_id'].astype(str).str[-6:]

fig, ax = plt.subplots(figsize=(11, 7))
x = np.arange(len(plot_df))
width = 0.35

ax.bar(x - width/2, plot_df['pca_noise_stability_fraction'], width, label='PCA DBSCAN small-grid noise fraction')
ax.bar(x + width/2, plot_df['umap_noise_stability_fraction'], width, label='UMAP DBSCAN small-grid noise fraction')

ax.set_xticks(x)
ax.set_xticklabels(plot_df['short_source_id'], rotation=90)
ax.set_ylim(0, 1.05)
ax.set_xlabel('Project 1 candidate source_id suffix')
ax.set_ylabel('Noise stability fraction across 9 small-grid DBSCAN settings')
ax.set_title('Candidate-level cross-method DBSCAN noise evidence')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_FIG, dpi=200)
plt.show()

print('Saved:', OUT_FIG)

## Interpretation note

The PCA small-grid results quantify whether Project 1 candidates repeatedly occupy sparse/outlier regions in the PCA representation. The UMAP small-grid results provide a nonlinear-embedding comparison. Objects with high PCA stability but low UMAP stability should be interpreted as representation-dependent feature-space outliers, not as independently confirmed astrophysical substructures.